In [0]:
from pyspark.sql import functions as F, Window

In [0]:
BRONZE = "abfss://basecontainer@ecommprojadls.dfs.core.windows.net/bronze/orders/"

src = spark.read.parquet(BRONZE)

src = src.withColumn("_op_rank", F.when(F.col("_op") == "D", 2).otherwise(1))

w = Window.partitionBy("order_id").orderBy(
    F.col("_change_version").desc(),
    F.col("_op_rank").desc()
)

latest = (src
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn", "_op_rank"))

print(f"{src.count():,} bronze rows → {latest.count():,} unique keys")

In [0]:
from delta.tables import DeltaTable

tgt = DeltaTable.forName(spark, "ecomm.silver.orders_current")

(tgt.alias("t")
   .merge(latest.alias("s"), "t.order_id = s.order_id")

   .whenMatchedUpdate(
       condition="s._change_version > t._change_version AND s._op != 'D'",
       set={
           "customer_unique_id": "s.customer_unique_id",
           "order_status": "s.order_status",
           "order_purchase_timestamp": "s.order_purchase_timestamp",
           "order_approved_at": "s.order_approved_at",
           "order_delivered_carrier_date": "s.order_delivered_carrier_date",
           "order_delivered_customer_date": "s.order_delivered_customer_date",
           "order_estimated_delivery_date": "s.order_estimated_delivery_date",
           "updated_at": "s.updated_at",
           "_change_version": "s._change_version",
           "_is_deleted": F.lit(False),
           "_merged_at": F.current_timestamp(),
       })

   .whenMatchedUpdate(
       condition="s._change_version > t._change_version AND s._op = 'D'",
       set={"_is_deleted": F.lit(True),
            "_change_version": "s._change_version",
            "_merged_at": F.current_timestamp()})

   .whenNotMatchedInsert(values={
       "order_id": "s.order_id",
       "customer_unique_id": "s.customer_unique_id",
       "order_status": "s.order_status",
       "order_purchase_timestamp": "s.order_purchase_timestamp",
       "order_approved_at": "s.order_approved_at",
       "order_delivered_carrier_date": "s.order_delivered_carrier_date",
       "order_delivered_customer_date": "s.order_delivered_customer_date",
       "order_estimated_delivery_date": "s.order_estimated_delivery_date",
       "updated_at": "s.updated_at",
       "_change_version": "s._change_version",
       "_is_deleted": F.expr("s._op = 'D'"),
       "_merged_at": F.current_timestamp(),
   })
   .execute())

In [0]:
before = spark.table("ecomm.silver.orders_current")
snap = before.agg(F.count("*"), F.sum("_change_version"),
                  F.sum(F.col("_is_deleted").cast("int"))).collect()[0]

print(snap)